In [ ]:
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)


In [ ]:
BASE_DIR = Path.cwd().parent
DATA_DIR = BASE_DIR / "data" / "processed_data"

train_df = pd.read_parquet(DATA_DIR / "train_processed.parquet")
valid_df = pd.read_parquet(DATA_DIR / "valid_processed.parquet")

print(train_df.shape, valid_df.shape)


In [ ]:
TARGET_COL = "target"
DROP_COLS = ["target","datetime","date","is_consumption","row_id"]

FEATURE_COLS = [c for c in train_df.columns if c not in DROP_COLS]
print("num features:", len(FEATURE_COLS))


In [ ]:
train_df = train_df.sort_values(["prediction_unit_id","is_consumption","datetime"])
valid_df = valid_df.sort_values(["prediction_unit_id","is_consumption","datetime"])

train_cons = train_df[train_df.is_consumption==1].copy()
valid_cons = valid_df[valid_df.is_consumption==1].copy()

train_prod = train_df[train_df.is_consumption==0].copy()
valid_prod = valid_df[valid_df.is_consumption==0].copy()

print(train_cons.shape, train_prod.shape)


In [ ]:
scaler_cons = StandardScaler()
scaler_prod = StandardScaler()

train_cons[FEATURE_COLS] = scaler_cons.fit_transform(train_cons[FEATURE_COLS])
valid_cons[FEATURE_COLS] = scaler_cons.transform(valid_cons[FEATURE_COLS])

train_prod[FEATURE_COLS] = scaler_prod.fit_transform(train_prod[FEATURE_COLS])
valid_prod[FEATURE_COLS] = scaler_prod.transform(valid_prod[FEATURE_COLS])


In [ ]:
SEQ_LEN = 24

class SequenceDataset(Dataset):
    def __init__(self, df, feature_cols, seq_len=24):
        X_list=[]
        y_list=[]
        for pid, g in df.groupby("prediction_unit_id"):
            g=g.sort_values("datetime")
            X=g[feature_cols].values.astype(np.float32)
            y=g["target"].values.astype(np.float32)
            for i in range(seq_len,len(g)):
                X_list.append(X[i-seq_len:i])
                y_list.append(y[i])
        self.X=np.array(X_list)
        self.y=np.array(y_list)

    def __len__(self):
        return len(self.X)

    def __getitem__(self,idx):
        return torch.tensor(self.X[idx]), torch.tensor(self.y[idx])


In [ ]:
train_cons_dataset = SequenceDataset(train_cons,FEATURE_COLS,SEQ_LEN)
valid_cons_dataset = SequenceDataset(valid_cons,FEATURE_COLS,SEQ_LEN)
train_prod_dataset = SequenceDataset(train_prod,FEATURE_COLS,SEQ_LEN)
valid_prod_dataset = SequenceDataset(valid_prod,FEATURE_COLS,SEQ_LEN)

print(len(train_cons_dataset),len(train_prod_dataset))


In [ ]:
BATCH_SIZE=256

train_cons_loader=DataLoader(train_cons_dataset,batch_size=BATCH_SIZE,shuffle=True)
valid_cons_loader=DataLoader(valid_cons_dataset,batch_size=BATCH_SIZE,shuffle=False)

train_prod_loader=DataLoader(train_prod_dataset,batch_size=BATCH_SIZE,shuffle=True)
valid_prod_loader=DataLoader(valid_prod_dataset,batch_size=BATCH_SIZE,shuffle=False)


In [ ]:
class GRURegressor(nn.Module):
    def __init__(self,input_dim,hidden_dim=64):
        super().__init__()
        self.gru=nn.GRU(input_dim,hidden_dim,num_layers=2,batch_first=True,dropout=0.2)
        self.fc=nn.Sequential(
            nn.Linear(hidden_dim,64),
            nn.ReLU(),
            nn.Linear(64,1)
        )

    def forward(self,x):
        out,_=self.gru(x)
        out=out[:,-1,:]
        return self.fc(out).squeeze(1)


In [ ]:
input_dim=len(FEATURE_COLS)

cons_model=GRURegressor(input_dim).to(device)
prod_model=GRURegressor(input_dim).to(device)

cons_optimizer=torch.optim.Adam(cons_model.parameters(),lr=5e-4)
prod_optimizer=torch.optim.Adam(prod_model.parameters(),lr=5e-4)

criterion=nn.L1Loss()


In [ ]:
def train_epoch(model,loader,opt):
    model.train()
    total=0
    for X,y in loader:
        X=X.to(device);y=y.to(device)
        opt.zero_grad()
        pred=model(X)
        loss=criterion(pred,y)
        loss.backward()
        opt.step()
        total+=loss.item()*X.size(0)
    return total/len(loader.dataset)

def evaluate(model,loader):
    model.eval()
    preds=[];targets=[]
    with torch.no_grad():
        for X,y in loader:
            X=X.to(device)
            p=model(X).cpu().numpy()
            preds.append(p);targets.append(y.numpy())
    preds=np.concatenate(preds)
    targets=np.concatenate(targets)
    return mean_absolute_error(targets,preds)


In [ ]:
EPOCHS=8

for epoch in range(EPOCHS):
    train_loss=train_epoch(cons_model,train_cons_loader,cons_optimizer)
    val_mae=evaluate(cons_model,valid_cons_loader)
    print("CONS",epoch+1,train_loss,val_mae)


In [ ]:
for epoch in range(EPOCHS):
    train_loss=train_epoch(prod_model,train_prod_loader,prod_optimizer)
    val_mae=evaluate(prod_model,valid_prod_loader)
    print("PROD",epoch+1,train_loss,val_mae)
